# 23 Player-Season Aggregate Baseline

Purpose: predict player-season aggregate targets from mechanics-prior embeddings. This covers BBE-derived season stats such as average EV, average xBA/xwOBA, hard-hit rate, and barrel rate. BA/OPS/OBP/SLG are evaluated when `manifests/player_season_batting_v1.parquet` exists; otherwise they remain explicitly unavailable.

Runtime: CPU is enough.

Required inputs:
- `manifests/bbe_events_v1.parquet`
- `features/{player_season_embedding_feature_id}/manifest.parquet` from notebook 13
- optional `manifests/player_season_batting_v1.parquet` from notebook 11_f for BA/OPS/OBP/SLG labels

Outputs:
- `predictions/{player_season_run_id}/predictions_v1.parquet`
- `predictions/{player_season_run_id}/metrics_v1.json`
- `datasets/player_season_targets/{player_season_run_id}/manifest.parquet`
- `reports/preflight/player_season_aggregate_{player_season_run_id}.json`

Next: run notebook 15 so fusion can include both event-level and player-season predictions.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path(os.environ['BATTING_CODE_ROOT'])
    mydrive = Path('/content/drive/MyDrive')
    candidates = [
        fixed,
        Path('/content/drive/MyDrive/codex/batting_codex_handoff'),
        Path('/content/drive/MyDrive/batting_codex_handoff'),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            return candidate
    raise FileNotFoundError(
        'sport_pipeline repo が見つかりません。Drive mount と repo 配置を確認してください.\n'
        f'BATTING_CODE_ROOT={fixed}\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別配置の場合は notebook の最初の cell より前に次を実行してください.\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/あなたの配置/batting_codex_handoff\n'
        f'MyDrive exists={mydrive.exists()}\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()
REPO_DIR = _resolve_repo_dir()
os.environ['BATTING_CODE_ROOT'] = str(REPO_DIR)
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME
src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from sport_pipeline.pipeline.run_profile import artifact_id, load_run_profile, resolve_statcast_date_range, run_id

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
PLAYER_SEASON_RUN_ID = run_id(RUN_PROFILE, 'player_season_run_id', 'player_season_aggregate_mlb_2024_2026_v2')
PLAYER_SEASON_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'player_season_embedding_feature_id')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('BASE_DIR =', BASE_DIR)
print('PLAYER_SEASON_RUN_ID =', PLAYER_SEASON_RUN_ID)
print('PLAYER_SEASON_EMBEDDING_FEATURE_ID =', PLAYER_SEASON_EMBEDDING_FEATURE_ID)


In [ ]:
from sport_pipeline.artifact_check import check_artifacts

required = [
    'manifests/bbe_events_v1.parquet',
    f'features/{PLAYER_SEASON_EMBEDDING_FEATURE_ID}/manifest.parquet',
]
optional = ['manifests/player_season_batting_v1.parquet']
print('required:', check_artifacts(BASE_DIR, required))
print('optional:', check_artifacts(BASE_DIR, optional))


In [ ]:
from sport_pipeline.models.player_season import run_player_season_aggregate_baseline

outputs = run_player_season_aggregate_baseline(
    BASE_DIR,
    prediction_run_id=PLAYER_SEASON_RUN_ID,
    player_season_embedding_feature_id=PLAYER_SEASON_EMBEDDING_FEATURE_ID,
    require_non_empty=True,
)
for name, path in outputs.items():
    print(name, '->', path, 'exists=', path.exists())


In [ ]:
metrics_path = outputs['metrics']
summary_path = outputs['summary']
metrics = json.loads(metrics_path.read_text(encoding='utf-8')) if metrics_path.exists() else {}
summary = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.exists() else {}
print(json.dumps(summary, ensure_ascii=False, indent=2)[:4000])
print(json.dumps(metrics.get('metrics', {}), ensure_ascii=False, indent=2)[:4000])
print('BA/OPS/OBP/SLG labels require manifests/player_season_batting_v1.parquet from 11_f.')
print('NEXT: notebooks/15_full_fusion.ipynb, then 09_report_builder.ipynb and 22_research_outputs.ipynb')
